# Data Pipeline Demo

This notebook demonstrates the data pipeline for preparing sequences for deep learning models.

## Steps:
1. Generate sample telemetry data
2. Build sequences with sliding windows
3. Generate RUL labels
4. Split by engines (train/val/test)
5. Scale data
6. Validate and visualize

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app.data import SequenceBuilder, TimeSeriesScaler, handle_missing_data, validate_data

sns.set_style('darkgrid')
%matplotlib inline

## 1. Generate Sample Data

In [ ]:
# Generate sample data using the script
!python ../scripts/generate_sample_data.py \
    --output ../data/sample_telemetry.csv \
    --n-engines 30 \
    --failure-rate 0.4 \
    --min-cycles 80 \
    --max-cycles 200

In [ ]:
# Load data
data = pd.read_csv('../data/sample_telemetry.csv')
print(f"Loaded {len(data)} samples from {data['engine_id'].nunique()} engines")
data.head()

## 2. Validate Data

In [ ]:
# Validate data quality
validation_report = validate_data(data, label_col='label')

print("\nClass Distribution:")
print(validation_report['class_distribution'])
print(f"\nImbalance Ratio: {validation_report.get('imbalance_ratio', 'N/A'):.2f}")

## 3. Build Sequences

In [ ]:
# Initialize sequence builder
builder = SequenceBuilder(window_length=32, stride=4)

# Build sequences for classification
sequences, labels, engine_ids = builder.build_sequences(
    data,
    engine_id_col='engine_id',
    cycle_col='cycle',
    label_col='label'
)

print(f"\nSequences shape: {sequences.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Failure rate in sequences: {labels.mean():.2%}")

## 4. Generate RUL Labels

In [ ]:
# Generate RUL labels
data_with_rul = builder.generate_rul_labels(data, max_rul=125)

# Build RUL sequences
rul_sequences, rul_targets, rul_engine_ids = builder.build_rul_sequences(
    data_with_rul,
    engine_id_col='engine_id',
    cycle_col='cycle'
)

print(f"\nRUL Sequences shape: {rul_sequences.shape}")
print(f"RUL Targets shape: {rul_targets.shape}")
print(f"RUL range: [{rul_targets.min():.1f}, {rul_targets.max():.1f}]")

In [ ]:
# Visualize RUL distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(rul_targets, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('RUL (cycles)')
plt.ylabel('Frequency')
plt.title('RUL Distribution')

plt.subplot(1, 2, 2)
# Plot RUL over time for a few engines
for eid in data_with_rul['engine_id'].unique()[:5]:
    engine_data = data_with_rul[data_with_rul['engine_id'] == eid]
    plt.plot(engine_data['cycle'], engine_data['rul'], label=f'Engine {eid}', alpha=0.7)

plt.xlabel('Cycle')
plt.ylabel('RUL (cycles)')
plt.title('RUL Trajectories (Sample Engines)')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Split by Engines

In [ ]:
# Split sequences by engines
splits = builder.split_by_engines(
    sequences,
    labels,
    engine_ids,
    train_ratio=0.7,
    val_ratio=0.15,
    random_seed=42
)

X_train, y_train = splits['train']
X_val, y_val = splits['val']
X_test, y_test = splits['test']

print(f"\nTrain: {len(X_train)} sequences, {y_train.mean():.2%} failure rate")
print(f"Val:   {len(X_val)} sequences, {y_val.mean():.2%} failure rate")
print(f"Test:  {len(X_test)} sequences, {y_test.mean():.2%} failure rate")

## 6. Scale Data

In [ ]:
# Fit scaler on training data
scaler = TimeSeriesScaler(scaler_type='standard')
scaler.fit(X_train, feature_names=builder.feature_columns)

# Transform all splits
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nScaled data statistics:")
print(f"Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Val mean:   {X_val_scaled.mean():.4f}, std: {X_val_scaled.std():.4f}")
print(f"Test mean:  {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

## 7. Visualize Sequences

In [ ]:
# Plot a few sample sequences
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Plot healthy and failure sequences
healthy_idx = np.where(y_train == 0)[0][0]
failure_idx = np.where(y_train == 1)[0][0] if (y_train == 1).any() else healthy_idx

for i, (idx, label) in enumerate([(healthy_idx, 'Healthy'), (failure_idx, 'Failure')]):
    # Before scaling
    axes[i, 0].plot(X_train[idx, :, :4])  # First 4 features
    axes[i, 0].set_title(f'{label} Sequence (Raw)')
    axes[i, 0].set_xlabel('Time Step')
    axes[i, 0].set_ylabel('Value')
    
    # After scaling
    axes[i, 1].plot(X_train_scaled[idx, :, :4])
    axes[i, 1].set_title(f'{label} Sequence (Scaled)')
    axes[i, 1].set_xlabel('Time Step')
    axes[i, 1].set_ylabel('Scaled Value')

plt.tight_layout()
plt.show()

## 8. Save Processed Data

In [ ]:
# Save to npz files
import os
os.makedirs('../data/processed', exist_ok=True)

np.savez_compressed(
    '../data/processed/train.npz',
    X=X_train_scaled,
    y=y_train
)

np.savez_compressed(
    '../data/processed/val.npz',
    X=X_val_scaled,
    y=y_val
)

np.savez_compressed(
    '../data/processed/test.npz',
    X=X_test_scaled,
    y=y_test
)

# Save scaler
scaler.save('../data/processed/scaler.joblib')

print("✅ Saved processed data to data/processed/")

## Summary

We've successfully:
- Generated synthetic telemetry data
- Built sequences with sliding windows
- Generated RUL labels
- Split data by engines (no leakage)
- Scaled features
- Saved processed data for model training

Next steps:
- Train autoencoder for anomaly detection
- Train LSTM for sequence classification
- Train RUL regression model